# Sprint 1 Day 2

## Objective

Create reusable data transformation functions.

## Goals

- Standardize company IDs
- Extract financial years
- Identify reporting types
- Prepare datasets for database loading

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.append(str(project_root))

print(project_root)

c:\Users\rishi\n100-financial-intelligence


In [3]:
from src.etl.extract import load_all_datasets
from src.etl.transform import (
    normalize_company_id,
    extract_financial_year
)

datasets = load_all_datasets()

pnl = datasets["profitandloss.xlsx"]

pnl = normalize_company_id(pnl)
pnl = extract_financial_year(pnl)

pnl[
    ["company_id", "year", "financial_year"]
].head(10)

,company_id,year,financial_year
0,ABB,Dec 2012,2012
1,ABB,Mar 2014,2014
2,ABB,Mar 2015,2015
3,ABB,Mar 2016,2016
4,ABB,Mar 2017,2017
5,ABB,Mar 2018,2018
6,ABB,Mar 2019,2019
7,ABB,Mar 2020,2020
8,ABB,Mar 2021,2021
9,ABB,Mar 2022,2022


In [4]:
pnl[
    pnl["year"].str.contains(
        "TTM|9m|15",
        case=False,
        na=False
    )
][["company_id", "year"]]

,company_id,year
2,ABB,Mar 2015
12,ABB,TTM
14,ADANIENSOL,Mar 2015
24,ADANIENSOL,TTM
27,ADANIENT,Mar 2015
...,...,...
1249,ZOMATO,TTM
1252,ZYDUSLIFE,Mar 2015
1262,ZYDUSLIFE,TTM
1265,INDIGO,Mar 2015


In [5]:
pnl[
    pnl["year"].str.contains(
        "TTM|9m",
        case=False,
        na=False
    )
][["company_id", "year"]]

,company_id,year
12,ABB,TTM
24,ADANIENSOL,TTM
37,ADANIENT,TTM
46,ADANIGREEN,TTM
59,ADANIPORTS,TTM
...,...,...
1228,VEDL,TTM
1241,WIPRO,TTM
1249,ZOMATO,TTM
1262,ZYDUSLIFE,TTM


In [6]:
def get_reporting_type(value):

    value = str(value)

    if "TTM" in value:
        return "TTM"

    elif "9m" in value.lower():
        return "9M"

    else:
        return "Annual"

In [7]:
pnl["reporting_type"] = pnl["year"].apply(
    get_reporting_type
)

pnl[
    ["year", "financial_year", "reporting_type"]
].head(20)

,year,financial_year,reporting_type
0,Dec 2012,2012,Annual
1,Mar 2014,2014,Annual
2,Mar 2015,2015,Annual
3,Mar 2016,2016,Annual
4,Mar 2017,2017,Annual
5,Mar 2018,2018,Annual
6,Mar 2019,2019,Annual
7,Mar 2020,2020,Annual
8,Mar 2021,2021,Annual
9,Mar 2022,2022,Annual


## Reporting Type Classification

The original year field contains multiple reporting formats.

Examples:

- Annual Reports
  - Mar 2024
  - Dec 2023
  - Sep 2022

- TTM Reports
  - TTM

- Partial Period Reports
  - Mar 2016 9m

A reporting_type field was created to preserve this information while enabling standardized year analysis.

## SQLite Database Creation

Objective:
Store all project datasets in a centralized database.

Result:
Successfully loaded 12 datasets into SQLite.

Benefits:
- Faster querying
- Easier joins
- Centralized storage
- Supports KPI engine development

## Financial Concept - ROE vs ROCE

ROE (Return on Equity):
Measures how efficiently a company uses shareholder funds.

ROCE (Return on Capital Employed):
Measures how efficiently a company uses all capital, including debt.

Investment Insight:
Companies with both high ROE and high ROCE are often considered high-quality businesses because they generate strong returns without excessive capital requirements.

## Data Quality Rule 3 Findings

Duplicate company-year combinations were identified across several datasets.

Observation:
The duplicates appear systematic rather than random, suggesting a data ingestion or source duplication issue.

Next Step:
Compare duplicate rows directly before deciding whether records should be removed or retained.